In [3]:
# Inicialización
from ingest import load_faq_data
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [4]:
# Usamos ID como etiqueta del documento en las pruebas de evaluación
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

9e508f2212
Course: When does the course start?
A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).

- Register via the link in the course repo before the cohort starts.
- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.
- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

# Usado para enviar al prompt la lista de preguntas y respuestas y generar preguntas de ejemplo.
user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
# Enviamos el prompt y usamos el parámetro text_format para pedir una estructura específica
# gracias a la clase questions creada con pydantic
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)

questions=['When does the next data engineering zoomcamp cohort usually start each year?', 'Where can I find the exact start date for the current cohort?', 'How do I register for the course before it begins?', 'Is there a link in the repo for the current cohort signup and schedule?', 'What are the official channels for course announcements and updates?']


In [12]:
print(result.questions)


['When does the next data engineering zoomcamp cohort usually start each year?', 'Where can I find the exact start date for the current cohort?', 'How do I register for the course before it begins?', 'Is there a link in the repo for the current cohort signup and schedule?', 'What are the official channels for course announcements and updates?']


In [13]:
from evaluation_utils import llm_structured


In [14]:
# EN lugar de llamar directamente a openai, utiliamos llm_structured como envoltorio
# el cual es capaz de devolver no solo el resultado que necesitamos, sino el uso.
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['When does the next data engineering cohort usually begin, and where can I find the exact date?', 'How do I register for the current data engineering cohort before it starts?', 'Where is the official link for the course start date and signup page?', 'Is the data engineering zoomcamp run every year, and around what months does it happen?', 'What’s the best way to get updates about the course start and registration?']


In [15]:
usage.input_tokens, usage.output_tokens

(297, 96)

In [16]:
from evaluation_utils import calc_price


In [17]:
# El coste nos permite saber en las evaluaciones cuanto cuesta cada query
# y además nos permite saber incluso el coste de la propia generación de preguntas.
cost = calc_price(usage)

cost

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.000432,
 'total_cost': 0.00065475}

In [18]:
# Conversión de respuestas a registros de verdad (ground truth).
# Estas serán las preguntas que hagamos en las evaluaciones.
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'When does the next data engineering cohort usually begin, and where can I find the exact date?',
  'document': '9e508f2212'},
 {'question': 'How do I register for the current data engineering cohort before it starts?',
  'document': '9e508f2212'},
 {'question': 'Where is the official link for the course start date and signup page?',
  'document': '9e508f2212'},
 {'question': 'Is the data engineering zoomcamp run every year, and around what months does it happen?',
  'document': '9e508f2212'},
 {'question': 'What’s the best way to get updates about the course start and registration?',
  'document': '9e508f2212'}]

In [19]:
# Función para obtener el "ground truth" en los
# documentos obtenidos
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [21]:
# Ejemplo de procesar los documentos en un for, 
# donde se ve que el proceso es lento.
from evaluation_utils import llm_structured_retry

from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [32]:
filtered_documents = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
filtered_documents

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer':

In [33]:
# Ejemplo de procesamiento paralelo.
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

# Lo quehace "with" es gestionar correctamente recursos. 
# Se encarga de abstraer al programador de escribir los pasos necesarios para manejar un fichero o concurrencia.
# Esto se encarga de cerrar ficheros, terminar concurrencia, etc, sin necesidad de llamar explícitamente a los métodos.
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, filtered_documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [34]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [36]:
ground_truth

[{'question': 'I just found this course — am I still able to join now?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get the certificate somehow?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I join now?',
  'document': '74eb249bbf'},
 {'question': 'Can I still submit the project and get certified if I discovered the course late?',
  'document': '74eb249bbf'},
 {'question': 'I signed up for LLM Zoomcamp, but when should the confirmation email actually arrive?',
  'document': '977bf7786c'},
 {'question': 'Do I need to wait for an acceptance email before I can start the course and send in homework?',
  'document': '977bf7786c'},
 {'question': 'Is there a list of registered students, or can anyone just begin learning and submit assignments while the form is still open?',
  'document': '977bf7786c'},


In [25]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

1.080080250000001

In [26]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

1.080080250000001

In [27]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [37]:
df_ground_truth.to_csv("data/ground_truth-llm-zoomcamp.csv", index=False)